In [13]:
%pip install segmentation-models-pytorch albumentations opencv-python pandas tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/homes/nfs/ben/miniconda3/envs/kaggle/lib/python3.12/site-packages/pip/__main__.py", line 22, in <module>
    from pip._internal.cli.main import main as _main
  File "/homes/nfs/ben/miniconda3/envs/kaggle/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 10, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/homes/nfs/ben/miniconda3/envs/kaggle/lib/python3.12/site-packages/pip/_internal/cli/autocompletion.py", line 9, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/homes/nfs/ben/miniconda3/envs/kaggle/lib/python3.12/site-packages/pip/_internal/cli/main_parser.py", line 4, in <module>
    import subprocess
  File "/homes/nfs/ben/miniconda3/envs/kaggle/lib/python3.12/subprocess.py", line 117, in <module>
    import selectors
  File "/homes/nfs/ben/miniconda3/env

In [ ]:
%pip install --upgrade ipywidgets widgetsnbextension jupyterlab_widgets -q

In [2]:
import os, cv2, torch, numpy as np, pandas as pd
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from torch import nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

In [5]:
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.fnames = sorted([f for f in os.listdir(img_dir) if f.endswith('.png')])
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = cv2.imread(os.path.join(self.img_dir, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = None

        if self.mask_dir:
            mask = cv2.imread(os.path.join(self.mask_dir, fname), cv2.IMREAD_GRAYSCALE)
        
        if self.transform:
            if mask is not None:
                augmented = self.transform(image=img, mask=mask)
                img, mask = augmented["image"], augmented["mask"]
            else:
                augmented = self.transform(image=img)
                img = augmented["image"]

        return (img, mask.long()) if mask is not None else (img, fname)

In [8]:
train_tf = A.Compose([
    A.RandomCrop(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(p=0.3),
    A.ColorJitter(p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

test_tf = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

In [ ]:
train_img_dir = "./data/train/imgs"
train_mask_dir = "./data/train/masks"
test_img_dir = "./data/test/imgs"

train_set = SegDataset(train_img_dir, train_mask_dir, transform=train_tf)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True, num_workers=2)
test_set = SegDataset(test_img_dir, transform=test_tf)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

In [10]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.DeepLabV3Plus(
    encoder_name="resnet101",
    encoder_weights="imagenet",
    classes=16,
    activation=None
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

In [11]:
def train_model(model, loader, optimizer, scheduler, criterion, epochs=20):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, masks in tqdm(loader):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        print(f"[Epoch {epoch+1:02d}] Loss: {total_loss/len(loader):.4f}")
    return model

In [14]:
model = train_model(model, train_loader, optimizer, scheduler, criterion, epochs=25)

  0%|          | 0/63 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 10.91 GiB of which 378.06 MiB is free. Including non-PyTorch memory, this process has 10.54 GiB memory in use. Of the allocated memory 9.75 GiB is allocated by PyTorch, and 79.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def rle_encode(mask):
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[:-1:2]
    return " ".join(str(x) for x in runs)


In [ ]:
model.eval()
records = []
with torch.no_grad():
    for imgs, fnames in tqdm(test_loader):
        imgs = imgs.to(DEVICE)
        preds = model(imgs).softmax(1).argmax(1).cpu().numpy()
        for pred, fname in zip(preds, fnames):
            row = {"img": fname}
            for c in range(16):
                m = (pred == c).astype(np.uint8)
                row[f"class_{c}"] = "none" if m.sum() == 0 else rle_encode(m)
            records.append(row)

sub = pd.DataFrame(records)
sub.to_csv("submission.csv", index=False)
print("✅ submission.csv saved!")

In [ ]:
!kaggle competitions submit -c 2025-ncku-ee-ml-16-classes-segmentation -f submission.csv -m "DeepLabV3+ 512x512 AdamW"

100%|██████████████████████████████████████| 11.8M/11.8M [00:02<00:00, 5.14MB/s]
Successfully submitted to 2025 NCKU EE ML UAV Segmentation